# Transfer learning et fine-tuning sans tout oublier

Ce notebook s'entraine sur le meme type de probleme que le **Home Task 1 officiel IOAI 2026** ("Operation Night Watch") : on a un modele deja entraine sur des classes A, et on veut lui apprendre de nouvelles classes B **sans detruire** ce qu'il sait deja faire sur A. Ce phenomene s'appelle l'**oubli catastrophique** (catastrophic forgetting).

On simule ca avec un petit MLP et des donnees synthetiques (pas besoin d'audio ou de GPU pour comprendre le mecanisme), le raisonnement est transferable a n'importe quel type de donnees (image, audio, texte).

Reference officielle : https://github.com/IOAI-official/IOAI-2026 (dossier `Home Task`, tache 1).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import copy

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Le probleme en une image

- **Etape 1** : un modele est entraine sur les classes A (ex: 16 sons "de base")
- **Etape 2** : on recoit de nouvelles donnees, classes B (ex: 13 nouveaux sons)
- **Probleme naif** : si on entraine juste le modele sur B, il "oublie" A. La loss sur A explose alors qu'on ne l'a meme pas touchee pendant l'entrainement sur B.

On va d'abord reproduire ce probleme, puis voir 3 strategies pour l'attenuer.

In [ ]:
# on cree un petit modele de classification, 8 classes au total
# classes 0-4 = "classes A" (deja apprises), classes 5-7 = "classes B" (nouvelles)

class Classifieur(nn.Module):
    def __init__(self, dim_entree=10, dim_cachee=32, nb_classes=8):
        super().__init__()
        self.tronc = nn.Sequential(
            nn.Linear(dim_entree, dim_cachee),
            nn.ReLU(),
            nn.Linear(dim_cachee, dim_cachee),
            nn.ReLU(),
        )
        self.tete = nn.Linear(dim_cachee, nb_classes)

    def forward(self, x):
        return self.tete(self.tronc(x))

def generer_donnees(classes, n_par_classe=40, dim=10, graine=0):
    g = torch.Generator().manual_seed(graine)
    X, y = [], []
    for c in classes:
        centre = torch.randn(dim, generator=g) * 3
        pts = centre + torch.randn(n_par_classe, dim, generator=g) * 0.5
        X.append(pts)
        y.append(torch.full((n_par_classe,), c, dtype=torch.long))
    return torch.cat(X), torch.cat(y)

X_A, y_A = generer_donnees(classes=[0, 1, 2, 3, 4], graine=1)
X_B, y_B = generer_donnees(classes=[5, 6, 7], graine=2)

print("donnees A:", X_A.shape, "classes:", y_A.unique().tolist())
print("donnees B:", X_B.shape, "classes:", y_B.unique().tolist())

In [ ]:
def entrainer(modele, X, y, epochs=100, lr=0.01, geler_tronc=False, geler_lignes_tete=None):
    """geler_lignes_tete: liste d'indices de classes dont la ligne de la tete
    ne doit pas etre mise a jour (utile pour vraiment proteger les anciennes classes,
    voir section 2bis plus bas)."""
    if geler_tronc:
        for p in modele.tronc.parameters():
            p.requires_grad = False
    params_a_optimiser = [p for p in modele.parameters() if p.requires_grad]
    optimizer = optim.Adam(params_a_optimiser, lr=lr)
    fonction_loss = nn.CrossEntropyLoss()
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = fonction_loss(modele(X), y)
        loss.backward()
        if geler_lignes_tete:
            with torch.no_grad():
                for c in geler_lignes_tete:
                    modele.tete.weight.grad[c].zero_()
                    modele.tete.bias.grad[c].zero_()
        optimizer.step()
    return loss.item()

def evaluer(modele, X, y):
    with torch.no_grad():
        preds = modele(X).argmax(dim=1)
        return (preds == y).float().mean().item()
# etape 1 : modele "production" entraine seulement sur A
modele_base = Classifieur()
loss_finale = entrainer(modele_base, X_A, y_A, epochs=150)
print("loss finale sur A:", loss_finale)
print("accuracy sur A juste apres entrainement:", evaluer(modele_base, X_A, y_A))

In [ ]:
# etape 2 (naive) : on continue l'entrainement seulement sur B
modele_naif = copy.deepcopy(modele_base)
entrainer(modele_naif, X_B, y_B, epochs=150)

print("apres fine-tuning naif sur B seulement:")
print("  accuracy sur B (nouvelles classes) :", evaluer(modele_naif, X_B, y_B))
print("  accuracy sur A (anciennes classes) :", evaluer(modele_naif, X_A, y_A))
# observe : l'accuracy sur A s'effondre. C'est l'oubli catastrophique.

### Strategie 1 : geler le tronc (feature extractor), n'entrainer que la tete
Idee : les premieres couches apprennent des features generales (contours, frequences de base...), on les garde figees et on n'adapte que la derniere couche.

In [ ]:
modele_gele = copy.deepcopy(modele_base)
entrainer(modele_gele, X_B, y_B, epochs=150, geler_tronc=True)

print("avec tronc gele, fine-tuning de la tete seulement:")
print("  accuracy sur B :", evaluer(modele_gele, X_B, y_B))
print("  accuracy sur A :", evaluer(modele_gele, X_A, y_A))
# resultat surprenant : A s'effondre QUAND MEME, alors que le tronc n'a pas bouge.
# explication : la cross-entropy compare TOUTES les classes entre elles (softmax).
# entrainer sur B seulement pousse les logits des classes A vers le bas, meme si
# les features (le tronc) qui les produisent n'ont pas change. Geler le tronc
# ne suffit pas si la tete reste partagee et entierement entrainable.

### Strategie 1bis : proteger explicitement les lignes des anciennes classes

Le correctif : on gele le tronc **ET** on empeche les lignes de la tete correspondant aux anciennes classes (A) de bouger, en mettant leur gradient a zero avant chaque `optimizer.step()`. Seules les lignes de la tete pour les nouvelles classes (B) restent entrainables.

In [ ]:
modele_gele_correct = copy.deepcopy(modele_base)
classes_A = y_A.unique().tolist()   # [0, 1, 2, 3, 4], ce sont les lignes a proteger

entrainer(modele_gele_correct, X_B, y_B, epochs=150, geler_tronc=True, geler_lignes_tete=classes_A)

print("avec tronc gele + lignes de la tete de A protegees:")
print("  accuracy sur B :", evaluer(modele_gele_correct, X_B, y_B))
print("  accuracy sur A :", evaluer(modele_gele_correct, X_A, y_A))
# resultat : mieux que le naif, mais toujours pas parfait. Les logits de A n'ont
# pas bouge (ni le tronc ni leur ligne de tete), mais les logits de B, eux, sont
# devenus tres confiants pendant l'entrainement. Si un logit de B depasse le
# logit (inchange) de la bonne classe A sur un exemple donne, l'argmax se trompe
# quand meme. Geler les parametres ne suffit pas a garantir l'absence de confusion
# avec les nouvelles classes, seul le replay corrige vraiment ce probleme.

### Strategie 2 : rejouer un peu de A pendant l'entrainement sur B (replay / rehearsal)

Idee : on entraine sur un melange de B (nouvelles donnees) et d'un petit sous-ensemble garde de A (comme le `train.csv` du Home Task officiel qui garde "quelques clips par classe"). C'est la strategie la plus efficace en pratique et la plus utilisee dans le vrai Home Task 1.

In [ ]:
modele_replay = copy.deepcopy(modele_base)

# on ne garde qu'un petit sous-ensemble de A, comme dans l'enonce officiel
# ("note how few training clips per class are left")
n_garde_par_classe = 5
X_A_reduit, y_A_reduit = [], []
for c in y_A.unique():
    idx = (y_A == c).nonzero(as_tuple=True)[0][:n_garde_par_classe]
    X_A_reduit.append(X_A[idx])
    y_A_reduit.append(y_A[idx])
X_A_reduit = torch.cat(X_A_reduit)
y_A_reduit = torch.cat(y_A_reduit)

X_melange = torch.cat([X_B, X_A_reduit])
y_melange = torch.cat([y_B, y_A_reduit])

entrainer(modele_replay, X_melange, y_melange, epochs=150)

print("avec replay (B + petit echantillon de A):")
print("  accuracy sur B :", evaluer(modele_replay, X_B, y_B))
print("  accuracy sur A (toutes les donnees, pas juste l'echantillon) :", evaluer(modele_replay, X_A, y_A))
# meilleur compromis : B est bien appris ET A est largement preserve

### Strategie 3 : learning rate plus petit + moins d'epochs sur B

Idee simple mais souvent negligee : un fine-tuning trop agressif (lr eleve, trop d'epochs) ecrase plus vite les poids existants. Reduire le lr limite l'amplitude des mises a jour.

In [ ]:
modele_lr_faible = copy.deepcopy(modele_base)
entrainer(modele_lr_faible, X_B, y_B, epochs=150, lr=0.0005)   # lr divise par 20

print("avec learning rate reduit:")
print("  accuracy sur B :", evaluer(modele_lr_faible, X_B, y_B))
print("  accuracy sur A :", evaluer(modele_lr_faible, X_A, y_A))
# generalement un compromis moyen : moins d'oubli qu'en naif, mais B moins bien appris
# qu'avec le replay, et l'effet depend beaucoup du nombre d'epochs

## Tableau recapitulatif a retenir

| Strategie | Accuracy sur B (nouveau) | Accuracy sur A (ancien) | Quand l'utiliser |
|---|---|---|---|
| Fine-tuning naif | haute | s'effondre (0%) | jamais en pratique, sert de baseline a battre |
| Geler le tronc seul | haute | s'effondre aussi (piege!) | insuffisant seul si la tete reste partagee et entrainable |
| Geler tronc + lignes de tete de A | haute | ameliore mais imparfait | mieux que naif, mais les nouvelles classes peuvent quand meme "voler" des predictions |
| Replay (melange A+B) | haute | largement preservee (proche de 100%) | **la strategie par defaut a essayer en premier** |
| Learning rate reduit | plus faible sur B | mieux preservee sur A | complement aux autres, rarement suffisant seul |

**Le vrai piege a retenir** : geler des parametres empeche seulement CES parametres de bouger, ca ne garantit pas que le modele final se comporte pareil sur les anciennes classes. La cross-entropy et le softmax comparent tous les logits entre eux ; si les logits des nouvelles classes deviennent tres confiants, ils peuvent depasser les logits (pourtant inchanges) des anciennes classes sur certains exemples, et l'argmax se trompe quand meme.

**Memotech** : "geler des poids protege ces poids, pas le comportement du modele". La seule strategie qui a vraiment garde une accuracy proche de 100% sur A dans nos tests est le **replay** : montrer un peu de l'ancien pendant qu'on apprend le nouveau.

## Exercice (10-15 min)

1. Combine la strategie "tronc gele" ET la strategie "replay" : geler le tronc, entrainer la tete sur un melange B + echantillon de A
2. Compare les 4 metriques (accuracy A et B) avec les strategies precedentes
3. Bonus : fais varier `n_garde_par_classe` (2, 5, 10, 20) et trace comment l'accuracy sur A evolue en fonction de la quantite de donnees "rejouees"

In [ ]:
# ecris ta solution ici



